In [12]:
# cell_1_summary_stats.py

import json
import pandas as pd
from datetime import datetime

def load_trades(json_path):
    with open(json_path, 'r') as f:
        raw = json.load(f)

    trades = []
    for offer_rune, listings in raw["Runes"].items():
        for entry in listings:
            if not entry.get("price"):
                continue
            updated_at = pd.to_datetime(entry['updated_at'], utc=True).tz_convert(None)
            base = {
                'offer_rune': offer_rune,
                'seller': entry['seller'],
                'offer_quantity': entry['quantity'],
                'updated_at': updated_at,
                'trade_type': 'AND' if len(entry['price']) > 1 else 'Single',
                'ask_items_count': len(entry['price'])
            }
            for ask in entry['price']:
                record = base.copy()
                record.update({
                    'ask_item': ask['name'],
                    'ask_quantity': ask['quantity']
                })
                trades.append(record)

    df = pd.DataFrame(trades)
    df['days_ago'] = (pd.Timestamp.now() - df['updated_at']).dt.days
    return df

def display_trade_summary(df):
    print("📊 TRADE SUMMARY STATISTICS")
    print("=" * 50)
    print(f"Total Entries: {len(df):,}")
    print(f"Unique Trades: {len(df.drop_duplicates(['offer_rune', 'seller', 'updated_at'])):,}")
    print(f"Unique Sellers: {df['seller'].nunique():,}")
    print(f"Offer Runes: {df['offer_rune'].nunique()}, Ask Items: {df['ask_item'].nunique()}")

    print("\n📈 Trade Type Breakdown:")
    print(df['trade_type'].value_counts().to_string())

    print("\n🎯 Top 10 Offered Runes:")
    print(df['offer_rune'].value_counts().head(10).to_string())

    print("\n💰 Top 10 Requested Items:")
    print(df['ask_item'].value_counts().head(10).to_string())

    recent = df[df['days_ago'] <= 7]
    print(f"\n⏰ Trades in Last 7 Days: {len(recent):,}")
    if not recent.empty:
        print("Top Recent Offers:")
        print(recent['offer_rune'].value_counts().head(5).to_string())

# ✅ Set your path
json_path = "/Users/buddy/Desktop/traderie/data/raw/raw_trades_pc_sc_nl.json"
df_trades = load_trades(json_path)
display_trade_summary(df_trades)

📊 TRADE SUMMARY STATISTICS
Total Entries: 4,163
Unique Trades: 3,395
Unique Sellers: 697
Offer Runes: 24, Ask Items: 64

📈 Trade Type Breakdown:
trade_type
Single    2808
AND       1355

🎯 Top 10 Offered Runes:
offer_rune
Jah Rune    336
Ist Rune    253
Hel Rune    235
Ber Rune    229
Mal Rune    210
Lo Rune     206
Ral Rune    195
Amn Rune    191
Ohm Rune    187
Ort Rune    175

💰 Top 10 Requested Items:
ask_item
Jah Rune               1365
Ist Rune                821
Lo Rune                 320
Ber Rune                269
Mal Rune                200
Vex Rune                152
Ohm Rune                148
Um Rune                 138
Hel Rune                131
Token of Absolution     128

⏰ Trades in Last 7 Days: 3,247
Top Recent Offers:
offer_rune
Jah Rune    336
Ist Rune    253
Hel Rune    235
Ber Rune    229
Mal Rune    210


In [16]:
# cell_2_trade_explorer.py

import ipywidgets as widgets
from IPython.display import display, clear_output

class SimpleRuneTradeExplorer:
    def __init__(self, df):
        self.df = df
        self._setup_filters()
    
    def _setup_filters(self):
        rune_order = [
            "Zod", "Cham", "Jah", "Ber", "Sur", "Lo", "Ohm", "Vex", "Gul", "Ist",
            "Mal", "Um", "Pul", "Lem", "Fal", "Ko", "Lum", "Io", "Hel", "Dol",
            "Shael", "Sol", "Amn", "Thul", "Ort", "Ral", "Tal", "Ith", "Eth",
            "Nef", "Tir", "Eld", "El"
        ]
        ordered_runes = ['All'] + [r for r in rune_order if r in self.df['offer_rune'].unique()]
        
        self.offer_filter = widgets.SelectMultiple(
            options=ordered_runes,
            value=['All'],
            description='Offer Runes:',
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='300px', height='120px')
        )

        self.ask_filter = widgets.SelectMultiple(
            options=['All'] + sorted(self.df['ask_item'].unique()),
            value=['All'],
            description='Ask Items:',
            layout=widgets.Layout(width='300px', height='120px')
        )

        self.trade_type_filter = widgets.SelectMultiple(
            options=['All', 'Single', 'AND'],
            value=['All'],
            description='Trade Type:'
        )

        self.days_filter = widgets.IntRangeSlider(
            value=[0, self.df['days_ago'].max()],
            min=0,
            max=self.df['days_ago'].max(),
            description='Days Ago:',
            layout=widgets.Layout(width='400px')
        )

        self.max_rows = widgets.IntSlider(
            value=50,
            min=10,
            max=500,
            step=10,
            description='Max Rows:'
        )

        self.search_button = widgets.Button(
            description='Search',
            button_style='primary',
            icon='search'
        )

        self.output = widgets.Output()
        self.search_button.on_click(self._on_search)

    def _get_filtered_data(self):
        df = self.df.copy()

        if 'All' not in self.offer_filter.value:
            df = df[df['offer_rune'].isin(self.offer_filter.value)]

        if 'All' not in self.ask_filter.value:
            df = df[df['ask_item'].isin(self.ask_filter.value)]

        if 'All' not in self.trade_type_filter.value:
            df = df[df['trade_type'].isin(self.trade_type_filter.value)]

        df = df[(df['days_ago'] >= self.days_filter.value[0]) &
                (df['days_ago'] <= self.days_filter.value[1])]

        return df

    def _on_search(self, _):
        with self.output:
            clear_output(wait=True)
            df = self._get_filtered_data()

            if df.empty:
                print("❌ No matching trades.")
                return

            grouped = df.groupby(['offer_rune', 'seller', 'updated_at']).agg({
                'offer_quantity': 'first',
                'trade_type': 'first',
                'ask_item': lambda x: ' + '.join(x),
                'days_ago': 'first'
            }).reset_index()

            for _, trade in grouped.head(self.max_rows.value).iterrows():
                print(f"🔸 {trade['offer_rune']} (x{trade['offer_quantity']}) by {trade['seller']}")
                print(f"   Wants: {trade['ask_item']}")
                print(f"   Type: {trade['trade_type']} | {trade['days_ago']} days ago")
                print(f"   Updated: {trade['updated_at'].strftime('%Y-%m-%d %H:%M')}\n")

    def display(self):
        filters = widgets.VBox([
            widgets.HTML("<h3>🔍 Filters</h3>"),
            widgets.HBox([self.offer_filter, self.ask_filter]),
            widgets.HBox([self.trade_type_filter, self.days_filter]),
            self.max_rows,
            self.search_button
        ])
        full_ui = widgets.VBox([
            widgets.HTML("<h1>🏺 Rune Trade Explorer</h1>"),
            filters,
            widgets.HTML("<hr>"),
            self.output
        ])
        display(full_ui)

# ✅ Initialize and display the explorer using df_trades
explorer = SimpleRuneTradeExplorer(df_trades)
explorer.display()

In [14]:
# rune_trade_explorer_part2.py
class RuneTradeExplorer:  # Only needed if this is a standalone cell
    def setup_gui(self):
        rune_order = [
            "Zod", "Cham", "Jah", "Ber", "Sur", "Lo", "Ohm", "Vex", "Gul", "Ist",
            "Mal", "Um", "Pul", "Lem", "Fal", "Ko", "Lum", "Io", "Hel", "Dol",
            "Shael", "Sol", "Amn", "Thul", "Ort", "Ral", "Tal", "Ith", "Eth",
            "Nef", "Tir", "Eld", "El"
        ]
        ordered_runes = ['All'] + [r for r in rune_order if r in self.trades_df['offer_rune'].unique()]
        
        self.offer_filter = widgets.SelectMultiple(
            options=ordered_runes,
            value=['All'],
            description='Offer Runes:',
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='300px', height='120px')
        )
        self.ask_filter = widgets.SelectMultiple(
            options=['All'] + sorted(self.trades_df['ask_item'].unique()),
            value=['All'],
            description='Ask Items:',
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='300px', height='120px')
        )
        self.trade_type_filter = widgets.SelectMultiple(
            options=['All', 'Single', 'AND'],
            value=['All'],
            description='Trade Type:',
            style={'description_width': 'initial'}
        )
        self.days_filter = widgets.IntRangeSlider(
            value=[0, self.trades_df['days_ago'].max()],
            min=0,
            max=self.trades_df['days_ago'].max(),
            description='Days Ago:',
            style={'description_width': 'initial'}
        )
        self.seller_search = widgets.Text(
            placeholder='Search seller name...',
            description='Seller:',
            style={'description_width': 'initial'}
        )
        self.view_mode = widgets.RadioButtons(
            options=['Summary Stats', 'Trade List', 'Charts', 'Price Analysis'],
            value='Summary Stats',
            description='View:',
            style={'description_width': 'initial'}
        )
        self.max_rows = widgets.IntSlider(
            value=50, min=10, max=500, step=10,
            description='Max Rows:',
            style={'description_width': 'initial'}
        )
        self.refresh_btn = widgets.Button(description='Refresh Data', button_style='primary', icon='refresh')
        self.export_btn = widgets.Button(description='Export CSV', button_style='success', icon='download')
        self.output = widgets.Output()

        self.refresh_btn.on_click(self._on_refresh)
        self.export_btn.on_click(self._on_export)

        for widget in [self.offer_filter, self.ask_filter, self.trade_type_filter, 
                       self.days_filter, self.seller_search, self.view_mode, self.max_rows]:
            widget.observe(self._on_filter_change, names='value')

In [18]:
import ipywidgets as widgets
from IPython.display import display, clear_output
from collections import Counter, defaultdict
import json
from datetime import datetime

class RuneTradeExplorer:
    def __init__(self, data_file_path=None):
        # Load data (using your fake data for demo, replace with actual file loading)
        if data_file_path:
            with open(data_file_path, 'r') as f:
                self.data = json.load(f)
        else:
            # Extended fake data for demonstration
            self.data = {
                "Runes": {
                    "Ist Rune": [
                        {
                            "seller": "player1",
                            "quantity": 1,
                            "updated_at": "2025-05-21T04:52:43.189Z",
                            "price": [{"name": "Pul Rune", "quantity": 2}]
                        },
                        {
                            "seller": "player2", 
                            "quantity": 1,
                            "updated_at": "2025-05-21T03:59:40.573Z",
                            "price": [{"name": "Um Rune", "quantity": 1}, {"name": "Mal Rune", "quantity": 1}]
                        },
                        {
                            "seller": "player3",
                            "quantity": 2,
                            "updated_at": "2025-05-21T05:10:20.123Z", 
                            "price": [{"name": "Perfect Amethyst", "quantity": 15}]
                        }
                    ],
                    "Um Rune": [
                        {
                            "seller": "player4",
                            "quantity": 3,
                            "updated_at": "2025-05-21T06:30:15.456Z",
                            "price": [{"name": "Ist Rune", "quantity": 1}]
                        },
                        {
                            "seller": "player5",
                            "quantity": 1,
                            "updated_at": "2025-05-21T07:45:30.789Z",
                            "price": [{"name": "Pul Rune", "quantity": 3}, {"name": "Fal Rune", "quantity": 1}]
                        }
                    ],
                    "Pul Rune": [
                        {
                            "seller": "player6",
                            "quantity": 5,
                            "updated_at": "2025-05-21T08:20:45.321Z",
                            "price": [{"name": "Ist Rune", "quantity": 1}]
                        }
                    ],
                    "Mal Rune": [
                        {
                            "seller": "player7",
                            "quantity": 1,
                            "updated_at": "2025-05-21T09:15:10.654Z",
                            "price": [{"name": "Um Rune", "quantity": 2}]
                        }
                    ],
                    "Jah Rune": [
                        {
                            "seller": "player8",
                            "quantity": 1, 
                            "updated_at": "2025-05-21T10:30:25.987Z",
                            "price": [{"name": "Ber Rune", "quantity": 1}, {"name": "Sur Rune", "quantity": 1}]
                        }
                    ]
                }
            }
        
        self.setup_analysis()
        self.create_widgets()
        self.setup_callbacks()
    
    def setup_analysis(self):
        """Pre-compute all trade statistics"""
        self.rune_stats = {}
        self.ask_counter = Counter()
        self.rune_pairs = defaultdict(int)
        self.and_trades = []
        
        # Analyze all trades
        for rune_name, listings in self.data["Runes"].items():
            total_trades = len([e for e in listings if e.get("price")])
            total_quantity = sum(e.get("quantity", 0) for e in listings if e.get("price"))
            single_trades = 0
            and_trade_count = 0
            
            for entry in listings:
                if not entry.get("price"):
                    continue
                    
                # Count ask items
                for item in entry["price"]:
                    self.ask_counter[item["name"]] += item.get("quantity", 1)
                
                # Categorize trade type
                if len(entry["price"]) == 1:
                    single_trades += 1
                    # Track pairs for single trades
                    ask_item = entry["price"][0]["name"]
                    pair_key = tuple(sorted([rune_name, ask_item]))
                    self.rune_pairs[pair_key] += 1
                else:
                    and_trade_count += 1
                    self.and_trades.append((rune_name, entry))
                    # Track all combinations in AND trades
                    ask_items = [item["name"] for item in entry["price"]]
                    for ask_item in ask_items:
                        pair_key = tuple(sorted([rune_name, ask_item]))
                        self.rune_pairs[pair_key] += 1
            
            self.rune_stats[rune_name] = {
                "total_trades": total_trades,
                "total_quantity": total_quantity,
                "single_trades": single_trades,
                "and_trades": and_trade_count
            }
    
    def create_widgets(self):
        """Create all UI widgets"""
        # Get sorted rune list by trade count
        rune_options = [("-- Select Rune --", None)]
        rune_options.extend([
            (f"{rune} ({stats['total_trades']} trades)", rune) 
            for rune, stats in sorted(self.rune_stats.items(), 
                                    key=lambda x: x[1]['total_trades'], reverse=True)
        ])
        
        # Main dropdowns
        self.rune1_dropdown = widgets.Dropdown(
            options=rune_options,
            description="Primary Rune:",
            style={'description_width': '120px'}
        )
        
        self.rune2_dropdown = widgets.Dropdown(
            options=[("-- Select Second Rune --", None)],
            description="Secondary Rune:",
            disabled=True,
            style={'description_width': '120px'}
        )
        
        # Filter options
        self.trade_type_filter = widgets.Dropdown(
            options=[
                ("All Trades", "all"),
                ("Single Item Trades", "single"), 
                ("AND Trades (Multi-item)", "and")
            ],
            value="all",
            description="Trade Type:",
            style={'description_width': '120px'}
        )
        
        # Output areas
        self.output = widgets.Output()
        self.trade_details = widgets.Output()
        
        # Control buttons
        self.reset_btn = widgets.Button(
            description="Reset Selection",
            button_style="warning",
            icon="refresh"
        )
        
        self.show_and_trades_btn = widgets.Button(
            description="Show All AND Trades",
            button_style="info",
            icon="list"
        )
    
    def setup_callbacks(self):
        """Setup all widget callbacks"""
        self.rune1_dropdown.observe(self.on_rune1_change, names='value')
        self.rune2_dropdown.observe(self.on_rune2_change, names='value')
        self.trade_type_filter.observe(self.on_filter_change, names='value')
        self.reset_btn.on_click(self.on_reset)
        self.show_and_trades_btn.on_click(self.on_show_and_trades)
    
    def on_rune1_change(self, change):
        """Handle primary rune selection"""
        rune1 = change['new']
        if not rune1:
            self.rune2_dropdown.options = [("-- Select Second Rune --", None)]
            self.rune2_dropdown.disabled = True
            self.update_output("Select a rune to view trade statistics.")
            return
        
        # Update summary
        self.update_rune_summary(rune1)
        
        # Update second dropdown with related runes
        related_runes = self.get_related_runes(rune1)
        if related_runes:
            options = [("-- Select Second Rune --", None)]
            options.extend([
                (f"{rune} ({count} connections)", rune) 
                for rune, count in related_runes
            ])
            self.rune2_dropdown.options = options
            self.rune2_dropdown.disabled = False
        else:
            self.rune2_dropdown.options = [("-- No related runes --", None)]
            self.rune2_dropdown.disabled = True
        
        # Reset second dropdown value
        self.rune2_dropdown.value = None
    
    def on_rune2_change(self, change):
        """Handle secondary rune selection"""
        rune1 = self.rune1_dropdown.value
        rune2 = change['new']
        
        if rune1 and rune2:
            self.update_pair_summary(rune1, rune2)
        elif rune1:
            self.update_rune_summary(rune1)
    
    def on_filter_change(self, change):
        """Handle trade type filter change"""
        rune1 = self.rune1_dropdown.value
        rune2 = self.rune2_dropdown.value
        
        if rune1 and rune2:
            self.update_pair_summary(rune1, rune2)
        elif rune1:
            self.update_rune_summary(rune1)
    
    def on_reset(self, b):
        """Reset all selections"""
        self.rune1_dropdown.value = None
        self.rune2_dropdown.value = None
        self.trade_type_filter.value = "all"
        self.update_output("Select a rune to view trade statistics.")
        self.clear_trade_details()
    
    def on_show_and_trades(self, b):
        """Show all AND trades summary"""
        self.show_and_trades_summary()
    
    def get_related_runes(self, rune):
        """Get runes that trade with the given rune"""
        related = Counter()
        
        # Check offers (rune being offered)
        for entry in self.data["Runes"].get(rune, []):
            if entry.get("price"):
                for item in entry["price"]:
                    if item["name"] in self.rune_stats:
                        related[item["name"]] += 1
        
        # Check asks (rune being requested)
        for other_rune, listings in self.data["Runes"].items():
            if other_rune != rune:
                for entry in listings:
                    if entry.get("price"):
                        for item in entry["price"]:
                            if item["name"] == rune:
                                related[other_rune] += 1
        
        return related.most_common(10)
    
    def update_rune_summary(self, rune):
        """Update output with single rune summary"""
        stats = self.rune_stats[rune]
        filter_type = self.trade_type_filter.value
        
        # Get filtered trades
        trades = self.get_filtered_trades(rune, filter_type)
        
        summary = f"📊 {rune} Trade Summary\n"
        summary += "=" * 40 + "\n"
        summary += f"Total Trades: {stats['total_trades']}\n"
        summary += f"Total Quantity: {stats['total_quantity']}\n"
        summary += f"Single Trades: {stats['single_trades']}\n"
        summary += f"AND Trades: {stats['and_trades']}\n"
        
        if filter_type != "all":
            summary += f"\nFiltered Trades ({filter_type}): {len(trades)}\n"
        
        self.update_output(summary)
        self.show_trade_details(trades, rune)
    
    def update_pair_summary(self, rune1, rune2):
        """Update output with rune pair summary"""
        pair_key = tuple(sorted([rune1, rune2]))
        connections = self.rune_pairs.get(pair_key, 0)
        
        # Get specific pair trades
        trades = self.get_pair_trades(rune1, rune2)
        filter_type = self.trade_type_filter.value
        filtered_trades = [t for t in trades if self.matches_filter(t, filter_type)]
        
        summary = f"🔗 {rune1} ↔ {rune2} Trade Summary\n"
        summary += "=" * 50 + "\n"
        summary += f"Total Connections: {connections}\n"
        summary += f"Direct Trades: {len(trades)}\n"
        
        if filter_type != "all":
            summary += f"Filtered Trades ({filter_type}): {len(filtered_trades)}\n"
        
        self.update_output(summary)
        self.show_trade_details(filtered_trades, f"{rune1} & {rune2}")
    
    def get_filtered_trades(self, rune, filter_type):
        """Get trades filtered by type"""
        trades = []
        for entry in self.data["Runes"].get(rune, []):
            if not entry.get("price"):
                continue
            if self.matches_filter((rune, entry), filter_type):
                trades.append((rune, entry))
        
        # Also check where rune is being asked for
        for other_rune, listings in self.data["Runes"].items():
            if other_rune != rune:
                for entry in listings:
                    if entry.get("price"):
                        for item in entry["price"]:
                            if item["name"] == rune:
                                if self.matches_filter((other_rune, entry), filter_type):
                                    trades.append((other_rune, entry))
        
        return trades
    
    def get_pair_trades(self, rune1, rune2):
        """Get trades involving both runes"""
        trades = []
        
        # Check rune1 offers for rune2
        for entry in self.data["Runes"].get(rune1, []):
            if entry.get("price"):
                for item in entry["price"]:
                    if item["name"] == rune2:
                        trades.append((rune1, entry))
                        break
        
        # Check rune2 offers for rune1  
        for entry in self.data["Runes"].get(rune2, []):
            if entry.get("price"):
                for item in entry["price"]:
                    if item["name"] == rune1:
                        trades.append((rune2, entry))
                        break
        
        return trades
    
    def matches_filter(self, trade, filter_type):
        """Check if trade matches filter criteria"""
        if filter_type == "all":
            return True
        
        rune, entry = trade
        price_count = len(entry.get("price", []))
        
        if filter_type == "single":
            return price_count == 1
        elif filter_type == "and":
            return price_count > 1
        
        return True
    
    def show_trade_details(self, trades, title):
        """Show detailed trade information"""
        self.trade_details.clear_output()
        with self.trade_details:
            if not trades:
                print(f"No trades found for {title}")
                return
            
            print(f"📋 Trade Details for {title}")
            print("=" * 60)
            
            for i, (offer_rune, entry) in enumerate(trades[:10], 1):  # Show first 10
                seller = entry.get("seller", "Unknown")
                qty = entry.get("quantity", 1)
                time = entry.get("updated_at", "Unknown")
                
                print(f"\n{i}. Seller: {seller}")
                print(f"   Offering: {qty}x {offer_rune}")
                print(f"   Asking for:", end="")
                
                if len(entry["price"]) == 1:
                    item = entry["price"][0]
                    print(f" {item['quantity']}x {item['name']}")
                else:
                    print(" (AND Trade)")
                    for item in entry["price"]:
                        print(f"     • {item['quantity']}x {item['name']}")
                
                print(f"   Updated: {time[:19].replace('T', ' ')}")
            
            if len(trades) > 10:
                print(f"\n... and {len(trades) - 10} more trades")
    
    def show_and_trades_summary(self):
        """Show summary of all AND trades"""
        self.output.clear_output()
        with self.output:
            print("🔄 AND Trades Summary (Multi-item requests)")
            print("=" * 60)
            
            if not self.and_trades:
                print("No AND trades found in dataset.")
                return
            
            print(f"Total AND trades: {len(self.and_trades)}\n")
            
            # Group by number of items requested
            by_item_count = defaultdict(list)
            for rune, entry in self.and_trades:
                item_count = len(entry["price"])
                by_item_count[item_count].append((rune, entry))
            
            for item_count in sorted(by_item_count.keys()):
                trades = by_item_count[item_count]
                print(f"📦 {item_count}-item requests: {len(trades)} trades")
                
                for rune, entry in trades[:3]:  # Show first 3 examples
                    print(f"   • {entry['quantity']}x {rune} for:", end="")
                    for item in entry["price"]:
                        print(f" {item['quantity']}x {item['name']}", end=" +")
                    print(f" (by {entry['seller']})")
                
                if len(trades) > 3:
                    print(f"   ... and {len(trades) - 3} more")
                print()
    
    def update_output(self, text):
        """Update main output area"""
        self.output.clear_output()
        with self.output:
            print(text)
    
    def clear_trade_details(self):
        """Clear trade details output"""
        self.trade_details.clear_output()
    
    def display(self):
        """Display the complete interface"""
        # Header
        header = widgets.HTML("<h2>🗡️ Diablo II Rune Trade Explorer</h2>")
        
        # Controls row
        controls = widgets.HBox([
            self.rune1_dropdown,
            self.rune2_dropdown,
            self.trade_type_filter
        ])
        
        # Buttons row
        buttons = widgets.HBox([
            self.reset_btn,
            self.show_and_trades_btn
        ])
        
        # Output areas
        outputs = widgets.VBox([
            widgets.HTML("<h3>📊 Summary</h3>"),
            self.output,
            widgets.HTML("<h3>📋 Trade Details</h3>"),
            self.trade_details
        ])
        
        # Main layout
        main_layout = widgets.VBox([
            header,
            controls,
            buttons,
            outputs
        ])
        
        # Initialize with welcome message
        self.update_output("Welcome to the Rune Trade Explorer!\nSelect a rune to view trade statistics.")
        
        display(main_layout)

# Usage
# To use with real data: explorer = RuneTradeExplorer("raw_trades_pc_sc_nl.json")
explorer = RuneTradeExplorer()  # Uses fake data for demo
explorer.display()